# RKLLM xcpp Template

Uses `dlopen()+dlsym()` because xcpp/ORC does not resolve RKLLM symbols directly.

In [1]:
#include <stddef.h>
#include "/home/milh/projects/rknn-llm/rkllm-runtime/Linux/librkllm_api/include/rkllm.h"
#include <dlfcn.h>
#include <iostream>
#include <string>

void* rkllm_lib = dlopen(
    "/home/milh/projects/rknn-llm/rkllm-runtime/Linux/librkllm_api/aarch64/librkllmrt.so",
    RTLD_NOW | RTLD_GLOBAL);

if (!rkllm_lib) {
    std::cout << dlerror() << std::endl;
} else {
    std::cout << "RKLLM library loaded" << std::endl;
}

RKLLM library loaded


In [2]:
using CreateDefaultParamFn = RKLLMParam (*)();
//using InitFn = int (*)(LLMHandle*, RKLLMParam*, RKLLMResultCallback);
using InitFn =
int (*)(LLMHandle*,
        RKLLMParam*,
        LLMResultCallback);
using RunFn = int (*)(LLMHandle, RKLLMInput*, RKLLMInferParam*, void*);
using DestroyFn = int (*)(LLMHandle);

auto rkllm_createDefaultParam_fn =
    reinterpret_cast<CreateDefaultParamFn>(
        dlsym(rkllm_lib, "rkllm_createDefaultParam"));

auto rkllm_init_fn =
    reinterpret_cast<InitFn>(
        dlsym(rkllm_lib, "rkllm_init"));

auto rkllm_run_fn = 
    reinterpret_cast<RunFn>(
        dlsym(rkllm_lib, "rkllm_run"));

auto rkllm_destroy_fn =
    reinterpret_cast<DestroyFn>(
        dlsym(rkllm_lib, "rkllm_destroy"));

std::cout << "Symbols loaded" << std::endl;

Symbols loaded


In [3]:
static int callback(
    RKLLMResult* result,
    void* userdata,
    LLMCallState state)
{
    if (result && result->text)
        std::cout << result->text << std::flush;

    if (state == RKLLM_RUN_FINISH)
        std::cout << "\n[FINISHED]\n";

    return 0;
}

In [4]:
LLMHandle handle = nullptr;

RKLLMParam param = rkllm_createDefaultParam_fn();

param.model_path =
"/home/milh/models/Qwen3-4B-w8a8-npu.rkllm";

param.max_context_len = 4096;
param.max_new_tokens = 256;

int ret = rkllm_init_fn(
    &handle,
    &param,
    callback);

std::cout << "init=" << ret << std::endl;

init=0
I rkllm: rkllm-runtime version: 1.2.3, rknpu driver version: 0.9.8, platform: RK3588
I rkllm: loading rkllm model from /home/milh/models/Qwen3-4B-w8a8-npu.rkllm
I rkllm: rkllm-toolkit version: 1.2.1b1, max_context_limit: 4096, npu_core_num: 3, target_platform: RK3588, model_dtype: W8A8
I rkllm: Enabled cpus: [4, 5, 6, 7]
I rkllm: Enabled cpus num: 4


In [5]:
std::string prompt = "Write a haiku about RK3588.";

RKLLMInput input{};
input.input_type = RKLLM_INPUT_PROMPT;
input.prompt_input = prompt.data();

RKLLMInferParam infer{};
infer.mode = RKLLM_INFER_GENERATE;
infer.keep_history = 1;

ret = rkllm_run_fn(
    handle,
    &input,
    &infer,
    nullptr);

std::cout << "\nrun=" << ret << std::endl;

RK3588, next-gen core,  
powerful, efficient, bright future—  
innovation reborn.
[FINISHED]

run=0


In [6]:
rkllm_destroy_fn(handle);